In [1]:
import sys
print(sys.executable)


h:\projects\ai_based\Agent-Factory\.venv\Scripts\python.exe


In [10]:
from fastapi import FastAPI, HTTPException, Header, Depends, Request
from pydantic import BaseModel
from dotenv import load_dotenv
import os

import requests
from typing import List, Dict, Optional
from src.utils.llmp_utils import llmp_call
from src.agents.kb_agent import KBAgent

class GenerateRequest(BaseModel):
    model: str
    system_prompt: str = ''
    prompt: str
    format: Optional[dict] = None
    image: Optional[str] = None
    tools: Optional[List[Dict]] = None
    src: str = None
    temperature: float = 0.5
    
class Agent0:
    
    def __init__(self, tools_desc, model, agents):
        
        
        from dotenv import load_dotenv
        from sentence_transformers import CrossEncoder
        
        load_dotenv()
        
        
        self.src = 'agent_0'
        self.model = model
        self.llmp_url = os.getenv("LLMP_URL")
        self.llmp_password = os.getenv("LLMP_PASSWORD")
        self.tools_desc = tools_desc
        self.cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

        self.knowledge_bases_desc = {'physics_kb':'a knowledge base with information related to physics',
              'mathematics_kb':'a knowledge base with information related to mathematics"'
              }
        print("Initializing Agents!")
        for agent in agents:
            if agent == 'kb_agent':
                self.kb_agent = KBAgent(self.knowledge_bases_desc,self.model)
        
    def llmp_call(self, prompt, system_prompt, model):
        """ 
        Call the LLMP API to generate a response
        All related to the call is processed here
        """
        
        headers = {
        "Content-Type": "application/json",
        "Authorization": self.llmp_password
    }
        
        # Construct request payload
        request_data = GenerateRequest(
        model=model,
        system_prompt=system_prompt,
        prompt=prompt,
        tools=None,
        src=self.src)
        
        payload = request_data.model_dump(exclude_none=True)

        try:
            response = requests.post(self.llmp_url, headers=headers, json=payload)
            response.raise_for_status()  # Raise an error for bad responses (4xx, 5xx)
            return response.json()
        except requests.exceptions.RequestException as e:
            print(f"Request failed: {e}")
            return None
        
    def agent_0_response(self, user_prompt, temperature=0.5):
        """ 
        Encompasses logic behind tool decision making
        Calls the llmp_call method to generate a response
        """
        
        system_prompt = "You are an assistant. Your only goal is to provide me the name of the tool I would need to get the answer to the prompt and nothing else."
        
        tools_description = "\n ".join([f"{key}: {value}" for key, value in self.tools_desc.items()])
        prompt = f"{user_prompt}\nwhich of the following tools would you use?\n {tools_description}"
        
        llmp_response = llmp_call(prompt, system_prompt, self.model, temperature,src = 'Agent 0')['message']['content']
        
        results = self.cross_encoder.predict([[llmp_response, tool] for tool in self.tools_desc.keys()])
        
        tool_scores = dict(zip(self.tools_desc.keys(), results))
        best_tool = max(tool_scores, key=tool_scores.get)
        
        
        return best_tool,user_prompt
        
        
    def agent_0_chat(self, user_prompt):
            """
            Logic behind tool activation.
            Sends to agent_0_response for tool decision.
            Activates tool.

            Args:
                user_prompt (str)
            """
            
            agent_0_response = self.agent_0_response(user_prompt)
            
            selected_tool = agent_0_response[0]
            print(f"Passing to: \n{selected_tool} !")
            
            if selected_tool == 'image_generator':
                
                from src.image_generator import ImageGeneratorAgent
                img_gen = ImageGeneratorAgent()
                img_gen.generate(user_prompt)
                
            if selected_tool == 'ingestion_pipeline':
                
                from src.pipelines.ingestion_pipeline import ingest_pipeline
                ingest_pipeline()
            
            if selected_tool == 'Knowledge Base Query Agent':                
                
                user_prompt_rag = user_prompt.strip('given my documents')
                self.kb_agent.kb_agent_chat(user_prompt_rag)
                
                
                
                
        

In [11]:
agent_0_tools_desc = {'conversational_agent':'a conversational assistant, designed to conduct simple conversations with users',
              'image_generator':'a tool that generates images based on a given prompt. Should be triggered by explicit calls like "generate me an image of"',
              'ingestion_pipeline':'a tool that ingests documents, triggered by command "trigger ingestion"',
              'Knowledge Base Query Agent':'a tool that generates answers based on documents, triggeres by command "given my documents,"'
              }


agent_0 = Agent0(agent_0_tools_desc, "llama3.2:latest",['kb_agent'])

# knowledge_bases_desc = {'physics_kb':'a knowledge base with information related to physics',
#               'mathematics_kb':'a knowledge base with information related to mathematics"'
#               }


# kb_agent = KBAgent(knowledge_bases_desc, "llama3.2:latest")



In [ ]:
user_prompt = "what is the use of integrals in series convergence test?"
response = kb_agent.kb_agent_chat(user_prompt)

In [12]:
user_prompt = "given my documents, what is the use of integrals in series convergence test?"
response = agent_0.agent_0_chat(user_prompt)

Passing to: 
Knowledge Base Query Agent !
intfloat/e5-large-v2
Retrieving embeddings...
Performing semantic search...
Unloading embeddings model...
Processing context...
Processing references...
Calling LLMP...
The final answer is: To determine whether a series converges or diverges.
📚 References:

📄 **Calculus. Stewart.pdf**
   📑 Pages: 21, 111, 113, 117, 126, 127



In [1]:
import warnings
warnings.filterwarnings('ignore')

In [6]:
from src.agent_0 import Agent0
agent_0_tools_desc = {'Adversary Agent':'an adversary agent that is used to act as an adversary to the users strategies. Triggered by command "Need an adversary"',
              'image_generator':'a tool that generates images based on a given prompt. Should be triggered by explicit calls like "generate me an image of"',
              'ingestion_pipeline':'a tool that ingests documents, triggered by command "trigger ingestion"',
              'Knowledge Base Query Agent':'a tool that generates answers based on documents, triggeres by command "given my documents,"'
              }


agent_0 = Agent0(agent_0_tools_desc, "deepseek-r1:7b",['kb_agent','adv_agent'])



Initializing Agents!
Agents are ready for your use!


In [2]:
user_prompt = "trigger ingestion"
agent_0.agent_0_chat(user_prompt)

Passing to: 
ingestion_pipeline !
Economics-of-Strategy.pdf already ingested with the same configuration.
Thinking Strategically -Avinash K.pdf already ingested with the same configuration.
zero to one Peter Thiel.pdf already ingested with the same configuration.
Calculus. Stewart.pdf already ingested with the same configuration.
Processing document: Carl von Clausewitz, Beatrice Heuser - On War.pdf
Creating chunks...
loading embeddings model...
Creating embeddings...
Unloading embeddings model...
Checking table...
Saving embeddings...
Carl von Clausewitz, Beatrice Heuser - On War.pdf has been ingested.
Processing document: Makers of modern strategy_ from Machiavelli to the nuclear age.pdf
Creating chunks...
loading embeddings model...
Creating embeddings...
Unloading embeddings model...
Checking table...
Saving embeddings...
Makers of modern strategy_ from Machiavelli to the nuclear age.pdf has been ingested.
Processing document: Philip Sabin - Simulating War _ Studying Conflict throu

In [7]:
user_prompt = "Need an adversary. Assume you are a military strategist playing the role of an adversary in a war game against me. Consider we are on open terrain. My move: I have my cavalry brigade making a pincer move on your forces. What is your move to counter mine?"
response = agent_0.agent_0_chat(user_prompt)

Passing to: 
Adversary Agent !
KB Agent
intfloat/e5-large-v2
Retrieving embeddings...
Performing semantic search...
Unloading embeddings model...
Processing context...
Processing references...
Calling LLMP...
<think>
Okay, so I'm trying to figure out what my counter-move would be if I received the message that my cavalry brigade has made a pincer move on their opponent's forces in an open terrain war game. Let me break this down step by step.

First, understanding the scenario: it's an open terrain, which means there are no natural barriers like hills or forests to hide behind. My side has a cavalry brigade making a pincer move. A pincer move typically involves two or more forces converging from opposite directions to surround and attack an enemy area. So, my opponent is likely trying to limit my options or create confusion.

Since it's open terrain, the opponent doesn't have many hiding places. That means they need to be proactive in their defense. My initial thought is that they shou

To counter a pincer move from your cavalry on my forces, consider the following structured approach:

1. **Positioning and Cover:**
   - **Cover Behind:** Position yourself behind one of your cavalry units to block their advance. This sets up a defensive position and disrupts their plan.

2. **Defensive Line Formation:**
   - **Set Up Defense:** Change your formation to create a defensive line behind the pincer move. This allows you to disrupt their plan effectively.

3. **Attack from Opposite Side:**
   - Once the pincer move merges into a single unit, shift into a defensive position and then attack from the opposite side. This disrupts their counters and positions you for an effective counterattack.

4. **Utilize Resources:**
   - Ensure sufficient resources (troops or cavalry) to form a defensive line and set up your attack, especially on open terrain where flexibility is available.

By positioning yourself strategically, forming a defensive line, and attacking from the opposite side, you can disrupt the enemy's counters while preparing for an effective counterattack.
📚 References:

📄 **Philip Sabin - Simulating War _ Studying Conflict through Simulation Games.pdf**
   📑 Pages: 123, 138, 268

📄 **Carl von Clausewitz, Beatrice Heuser - On War.pdf**
   📑 Pages: 107, 202, 203


In [4]:
print(response)

<think>
Okay, so I need to figure out what enthalpy is when we add heat to a system at constant pressure. Hmm, let me start by recalling what I know about thermodynamics and the partition function.

From the example given in Example 20.3, they defined enthalpy H as U + PV. That makes sense because enthalpy is related to internal energy (U) and work done (PV). Now, for an isobaric process, which means pressure is constant, I remember that the change in enthalpy is equal to the heat added at constant pressure.

Wait, so if we add heat Q at constant pressure, then the change in enthalpy ΔH should be equal to Q. That seems right because when you do work on the system, it's related to pressure and volume changes, but since pressure is constant here, all the work done is converted into heat.

Let me think about this step by step. When we add heat Q at constant pressure, the system does no expansion or compression (since P is constant), so all the heat goes into increasing the internal energy

<think>
Okay, so I need to figure out how to create an AI startup using the AI Strategy Simulator product as the initial product. Hmm, where do I even start?

First off, I guess I should understand what an AI Strategy Simulator is. From the text provided earlier, it seems like it's a tool that helps businesses model and test different strategies for their AI projects. So maybe my startup could offer this simulator to other companies.

Wait, but how do I turn that into an actual product? Do I need to build a platform where clients can input their specific scenarios and get tailored recommendations? That sounds plausible. But building such a platform would require some technical skills, right?

I should probably start by researching existing AI tools to see what's out there. Maybe there are gaps in the market that my simulator could fill. For example, perhaps many existing tools don't offer enough customization or predictive analytics.

Next, I need to define the problem my product solves. It seems like businesses often struggle with planning and executing AI projects due to uncertainty. My simulator could help them by providing a structured way to evaluate different strategies under various scenarios.

But how do I position this in the market? Maybe target companies that are early in their AI adoption, like startups or small businesses where cost is a concern. They might be looking for an affordable yet powerful tool to guide their AI initiatives without needing deep expertise themselves.

I also need to think about the development process. Do I have the resources to build it myself, or should I partner with someone who has experience in AI and product development? Maybe hiring developers and data scientists would be necessary. Also, UX designers will help make the tool user-friendly.

Marketing is another big piece. How do I get people aware of this simulator? Perhaps through case studies, webinars on AI strategy, or partnerships with existing AI consulting firms. Offering a free trial could attract initial users to try it out and provide valuable feedback.

Once the product is developed, launching it would require a marketing campaign that highlights its unique features: customizable scenarios, predictive analytics, easy-to-use interface tailored for strategic planning. Maybe targeting decision-makers in tech companies or venture capitalists who are interested in AI projects.

After launch, continuous improvement based on user feedback will be crucial. Collecting reviews and incorporating new features could keep the product updated with the latest trends in AI strategy.

Wait, but what about competition? There might already be tools that offer similar services. I need to differentiate my simulator somehow. Maybe by integrating real-time data or offering more granular analytics than competitors do.

Also, considering scalability is important. As the startup grows, can the product handle larger and more complex scenarios without performance issues?

I should also think about monetization. Should it be a subscription model with pay-per-use pricing, or offer a freemium version where basic features are free and advanced ones require payment? Maybe a combination of both could work.

Another thought: maybe offering educational resources alongside the simulator to help users understand how to best utilize AI in their projects. Tutorials or webinars on AI strategy could add value beyond just the tool itself.

I also need to consider the regulatory environment, especially if the AI applications involve sensitive data. Ensuring compliance with relevant laws and regulations will be necessary for both the product features and customer data handling.

Lastly, building a community around the simulator might help in spreading the word. User groups or forums where people discuss their experiences could create a loyal customer base.

Overall, it seems like creating an AI Strategy Simulator startup involves several steps: researching the market, developing a unique product, positioning it effectively, marketing it through various channels, and continuously improving based on feedback. It's a bit overwhelming, but breaking it down into these manageable steps makes it more approachable.
</think>

Creating an AI Strategy Simulator Startup:

1. **Product Development:**
   - Develop a platform that allows businesses to input specific scenarios and receive tailored AI strategy recommendations.
   - Utilize existing tools as a foundation while addressing market gaps by offering customization, predictive analytics, and a user-friendly interface.

2. **Market Positioning:**
   - Target early adopters of AI such as startups and small businesses where cost-effectiveness is crucial.
   - Differentiate the product by integrating real-time data, granular analytics, and unique features like real-time data integration or advanced analytics.

3. **Target Audience:**
   - Focus on decision-makers in tech companies, venture capitalists, and organizations looking to adopt AI strategies efficiently.

4. **Development Process:**
   - Partner with developers, data scientists, and UX designers for technical and user interface development.
   - Consider hiring a team experienced in AI and product development to ensure robust functionality.

5. **Marketing Strategy:**
   - Offer free trials to attract initial users and gather feedback.
   - Use case studies, webinars on AI strategy, and partnerships with AI consulting firms to enhance visibility.
   - Highlight unique features such as customizable scenarios, predictive analytics, and ease of use in marketing materials.

6. **Monetization Model:**
   - Implement a subscription-based pricing model with pay-per-use options or offer a freemium version for basic features with premium advanced features available via payment.

7. **Educational Resources:**
   - Provide tutorials, webinars, and educational content to help users effectively utilize AI in their projects beyond the simulator itself.

8. **Regulatory Compliance:**
   - Ensure compliance with relevant laws and regulations, especially regarding data handling for sensitive information.

9. **Community Building:**
   - Foster a community through user groups or forums to encourage word-of-mouth marketing and loyal customer retention.

10. **Continuous Improvement:**
    - Regularly update the product based on user feedback and market trends.
    - Incorporate new features and analytics tools as AI strategies evolve.

By following these steps, the AI Strategy Simulator Startup can effectively address the strategic planning needs of businesses while differentiating itself in a competitive market.
📚 References:

📄 **Thinking Strategically -Avinash K.pdf**
   📑 Pages: 244, 245, 246, 247

📄 **zero to one Peter Thiel.pdf**
   📑 Pages: 42, 43

📄 **Economics-of-Strategy.pdf**
   📑 Pages: 34, 35

1. Identify a problem or opportunity that can be addressed using AI technology.
2. Conduct market research to understand the competitive landscape and potential demand for your AI solution.
3. Assemble a team with diverse skills, including AI experts, software engineers, and business professionals.
4. Develop a minimum viable product (MVP) to test your AI solution and gather feedback from potential customers.
5. Iterate on your MVP based on user feedback and market demand.
6. Secure funding through investors, grants, or other sources to support further development and growth.
7. Build partnerships with complementary businesses or organizations to expand your reach and enhance your AI solution.
8. Continuously monitor and adapt to changes in the market and technology landscape to maintain a competitive edge.
📚 References:

📄 **zero to one Peter Thiel.pdf**
   📑 Pages: 75, 76, 80, 81, 105, 108, 155, 156, 157